# Import Libraries

In [178]:
import pandas as pd
import numpy as np

# Import processed data

In [192]:
df = pd.read_csv('data/processed/train_processed.csv')
df.head()

,Region,Number vmail messages,Total day charge,Total eve charge,Total intl calls,Total intl charge,Customer service calls,International plan,Churn
0,Midwest,25,45.07,16.78,3,2.70,1,No,False
1,Midwest,26,27.47,16.62,3,3.70,1,No,False
2,Northeast,0,41.38,10.30,5,3.29,0,No,False
3,Midwest,0,50.90,5.26,7,1.78,2,Yes,False
4,South,0,28.34,12.61,3,2.73,3,Yes,False


# Split the data into X(features) and y(labels)

In [193]:
X = df.drop('Churn', axis=1)
y = df['Churn'].astype(int)

In [194]:
num = df.select_dtypes(include=[np.number]).columns.tolist()
cat = df.select_dtypes(include=str).columns.tolist()

In [195]:
print(num)
print(cat)

['Number vmail messages', 'Total day charge', 'Total eve charge', 'Total intl calls', 'Total intl charge', 'Customer service calls']
['Region', 'International plan']


In [196]:
df_test = pd.read_csv('data/raw/test.csv')

In [197]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 667 entries, 0 to 666
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   667 non-null    str    
 1   Account length          667 non-null    int64  
 2   Area code               667 non-null    int64  
 3   International plan      667 non-null    str    
 4   Voice mail plan         667 non-null    str    
 5   Number vmail messages   667 non-null    int64  
 6   Total day minutes       667 non-null    float64
 7   Total day calls         667 non-null    int64  
 8   Total day charge        667 non-null    float64
 9   Total eve minutes       667 non-null    float64
 10  Total eve calls         667 non-null    int64  
 11  Total eve charge        667 non-null    float64
 12  Total night minutes     667 non-null    float64
 13  Total night calls       667 non-null    int64  
 14  Total night charge      667 non-null    float64
 15  

In [198]:
regions = {
    'ME':'Northeast', 'NH':'Northeast', 'VT':'Northeast', 'MA':'Northeast', 'RI':'Northeast', 'CT':'Northeast', 'NY':'Northeast', 'PA':'Northeast', 'NJ':'Northeast',
    'WI':'Midwest', 'MI':'Midwest', 'IL':'Midwest', 'IN':'Midwest', 'OH':'Midwest', 'ND':'Midwest', 'SD':'Midwest', 'NE':'Midwest', 'KS':'Midwest', 'MN':'Midwest', 'IA':'Midwest', 'MO':'Midwest',
    'DE':'South', 'MD':'South', 'DC':'South', 'VA':'South', 'WV':'South', 'NC':'South', 'SC':'South', 'GA':'South', 'FL':'South', 'KY':'South', 'TN':'South', 'MS':'South', 'AL':'South', 'AR':'South', 'LA':'South', 'OK':'South', 'TX':'South',
    'ID':'West', 'MT':'West', 'WY':'West', 'NV':'West', 'UT':'West', 'CO':'West', 'AZ':'West', 'NM':'West', 'AK':'West', 'WA':'West', 'OR':'West', 'CA':'West', 'HI':'West'
}

df_test['Region'] = df_test['State'].map(regions)
df_test.drop(columns=['State'], inplace=True)

In [199]:
X_test = df_test[['Region','Number vmail messages','Total day charge','Total eve charge','Total intl calls','Total intl charge','Customer service calls','International plan']]
y_test = df_test['Churn'].astype(int)

In [200]:
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, PowerTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from xgboost import XGBClassifier
from imblearn.combine import SMOTETomek
from sklearn.metrics import classification_report

# Logistic Regression

In [201]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num_transform", PowerTransformer(method='yeo-johnson'), num),
        ("num", RobustScaler(), num),
        ("cat", OneHotEncoder(drop="first"),cat),
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(class_weight='balanced',random_state=42))
    ]
)

In [202]:
scoring_metrics = ["recall", "precision", "f1","balanced_accuracy"]
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    pipeline, X,y, cv=cv_strategy, scoring=scoring_metrics, return_train_score=True
)

print("--- Stratified 5-Fold Evaluation ---")
print(f"{'METRIC':<12} | {'TRAIN SCORE (Mean ± Std)':<26} | {'VALIDATION SCORE (Mean ± Std)'}")
print("-" * 75)
for metric in scoring_metrics:
    train_mean = cv_results[f"train_{metric}"].mean()
    train_std = cv_results[f"train_{metric}"].std()

    test_mean = cv_results[f"test_{metric}"].mean()
    test_std = cv_results[f"test_{metric}"].std()

    print(
        f"{metric.upper():<12} | "
        f"{train_mean:.4f} (± {train_std:.4f}){' ':<10} | "
        f"{test_mean:.4f} (± {test_std:.4f})"
    )

--- Stratified 5-Fold Evaluation ---
METRIC       | TRAIN SCORE (Mean ± Std)   | VALIDATION SCORE (Mean ± Std)
---------------------------------------------------------------------------
RECALL       | 0.8009 (± 0.0093)           | 0.7963 (± 0.0194)
PRECISION    | 0.4075 (± 0.0070)           | 0.4104 (± 0.0394)
F1           | 0.5401 (± 0.0078)           | 0.5403 (± 0.0322)
BALANCED_ACCURACY | 0.8012 (± 0.0060)           | 0.7992 (± 0.0147)


In [203]:
pipeline.fit(X,y)
y_pred = pipeline.predict(X_test)

In [205]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.79      0.87       572
           1       0.39      0.82      0.53        95

    accuracy                           0.79       667
   macro avg       0.68      0.80      0.70       667
weighted avg       0.88      0.79      0.82       667



# XGBoost

In [206]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first"),cat),
    ],
    remainder='passthrough'
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("smotetemk", SMOTETomek(
                random_state=42,
            )),
        ("classifier",
            XGBClassifier(
              random_state=42, 
              max_depth=3,    
              eval_metric="logloss", 
            ))
    ]
)

In [207]:
scoring_metrics = ["recall", "precision", "f1","balanced_accuracy"]
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = cross_validate(
    pipeline, X,y, cv=cv_strategy, scoring=scoring_metrics, return_train_score=True
)

print("--- Stratified 5-Fold Evaluation ---")
print(f"{'METRIC':<12} | {'TRAIN SCORE (Mean ± Std)':<26} | {'VALIDATION SCORE (Mean ± Std)'}")
print("-" * 75)
for metric in scoring_metrics:
    train_mean = cv_results[f"train_{metric}"].mean()
    train_std = cv_results[f"train_{metric}"].std()

    test_mean = cv_results[f"test_{metric}"].mean()
    test_std = cv_results[f"test_{metric}"].std()

    print(
        f"{metric.upper():<12} | "
        f"{train_mean:.4f} (± {train_std:.4f}){' ':<10} | "
        f"{test_mean:.4f} (± {test_std:.4f})"
    )

--- Stratified 5-Fold Evaluation ---
METRIC       | TRAIN SCORE (Mean ± Std)   | VALIDATION SCORE (Mean ± Std)
---------------------------------------------------------------------------
RECALL       | 0.8318 (± 0.0083)           | 0.7630 (± 0.0215)
PRECISION    | 0.9663 (± 0.0045)           | 0.8779 (± 0.0342)
F1           | 0.8940 (± 0.0047)           | 0.8155 (± 0.0079)
BALANCED_ACCURACY | 0.9134 (± 0.0040)           | 0.8723 (± 0.0082)


In [208]:
pipeline.fit(X,y)
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.98      0.97       572
           1       0.84      0.75      0.79        95

    accuracy                           0.94       667
   macro avg       0.90      0.86      0.88       667
weighted avg       0.94      0.94      0.94       667



In [209]:
import joblib
joblib.dump(pipeline, 'churn_model.pkl')

['churn_model.pkl']

In [210]:
df['Region'].unique()

<StringArray>
['Midwest', 'Northeast', 'South', 'West']
Length: 4, dtype: str